# 03 — Centralized Baseline Model Analysis
Loads `baseline_mlp.pth` and evaluates it on the test set. Shows confusion matrix and per-class F1.

In [ ]:
import sys, pickle
import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.metrics import f1_score, confusion_matrix, classification_report

sys.path.insert(0, str(Path().resolve().parent))
from src.configs.paths import PREPROCESSED_DIR, MODELS_DIR, ARTIFACTS_DIR
from src.configs.config import CONFIG
from src.components.model.model import MLPClassifier, set_model_parameters

with open(PREPROCESSED_DIR / 'label_encoder.pkl', 'rb') as f:
    le = pickle.load(f)

# Load test set
test_path = PREPROCESSED_DIR / 'test_set.npz'
data = np.load(test_path)
keys = list(data.keys())
X_test = data['X_test'].astype(np.float32) if 'X_test' in keys else data['X'].astype(np.float32)
y_test = data['y_test'].astype(np.int64) if 'y_test' in keys else data['y'].astype(np.int64)

# Load centralized model
cfg = CONFIG['model']
model = MLPClassifier(cfg['input_dim'], cfg['hidden_dims'], cfg['num_classes'], cfg['dropout_rate'])
saved = torch.load(MODELS_DIR / 'baseline_mlp.pth', map_location='cpu', weights_only=False)
model.load_state_dict(saved.get('model_state_dict', saved))
model.eval()
print('Centralized model loaded.')

In [ ]:
# Run inference
with torch.no_grad():
    logits = model(torch.tensor(X_test))
    y_pred = torch.argmax(logits, dim=1).numpy()

macro_f1 = f1_score(y_test, y_pred, average='macro', zero_division=0)
acc = (y_pred == y_test).mean()
print(f'Test Accuracy : {acc:.4f}')
print(f'Macro F1-Score: {macro_f1:.4f}')

In [ ]:
# Per-class F1 bar chart
per_class_f1 = f1_score(y_test, y_pred, average=None, zero_division=0)
class_names = le.classes_

fig, ax = plt.subplots(figsize=(14, 5))
colors = ['tomato' if f < 0.5 else 'steelblue' for f in per_class_f1]
ax.bar(range(len(per_class_f1)), per_class_f1, color=colors, edgecolor='white')
ax.set_xticks(range(len(class_names)))
ax.set_xticklabels(class_names, rotation=60, ha='right', fontsize=8)
ax.set_ylabel('F1-Score')
ax.set_ylim(0, 1)
ax.axhline(macro_f1, color='black', linestyle='--', label=f'Macro F1={macro_f1:.4f}')
ax.set_title('Centralized Baseline — Per-Class F1-Score')
ax.legend()
plt.tight_layout()
plt.savefig(ARTIFACTS_DIR / 'plots' / 'centralized_per_class_f1.png', dpi=150)
plt.show()

In [ ]:
# Confusion matrix (normalized)
cm = confusion_matrix(y_test, y_pred, normalize='true')
fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(cm, cmap='Blues', ax=ax, xticklabels=class_names, yticklabels=class_names,
            linewidths=0.3, fmt='.2f', annot=len(class_names) <= 15)
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
ax.set_title('Centralized Baseline — Normalized Confusion Matrix')
plt.xticks(rotation=60, ha='right', fontsize=7)
plt.yticks(fontsize=7)
plt.tight_layout()
plt.savefig(ARTIFACTS_DIR / 'plots' / 'centralized_confusion_matrix.png', dpi=150)
plt.show()

In [ ]:
# Full classification report
print(classification_report(y_test, y_pred, target_names=class_names, zero_division=0))